In [2]:
import pandas as pd

df = pd.read_csv(r"D:\journey\EUPHORIA\project\data\churn.csv")

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())


Shape: (64374, 12)

Columns:
['CustomerID', 'Age', 'Gender', 'Tenure', 'Usage Frequency', 'Support Calls', 'Payment Delay', 'Subscription Type', 'Contract Length', 'Total Spend', 'Last Interaction', 'Churn']

First 5 rows:


,CustomerID,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Subscription Type,Contract Length,Total Spend,Last Interaction,Churn
0,1,22,Female,25,14,4,27,Basic,Monthly,598,9,1
1,2,41,Female,28,28,7,13,Standard,Monthly,584,20,0
2,3,47,Male,27,10,2,29,Premium,Annual,757,21,0
3,4,35,Male,9,12,5,17,Premium,Quarterly,232,18,0
4,5,53,Female,58,24,9,2,Standard,Annual,533,18,0


In [3]:
print("DATA TYPES")
print("=" * 50)
print(df.dtypes)

print("\n\nMISSING VALUES")
print("=" * 50)
print(df.isnull().sum())

print("\n\nDUPLICATE ROWS")
print("=" * 50)
print(df.duplicated().sum())

print("\n\nCHURN VALUES")
print("=" * 50)
print(df["Churn"].value_counts())

print("\n\nUNIQUE VALUES IN CATEGORICAL COLUMNS")
print("=" * 50)

for column in df.select_dtypes(include="object").columns:
    print(f"\n{column}:")
    print(df[column].unique())


DATA TYPES
CustomerID            int64
Age                   int64
Gender               object
Tenure                int64
Usage Frequency       int64
Support Calls         int64
Payment Delay         int64
Subscription Type    object
Contract Length      object
Total Spend           int64
Last Interaction      int64
Churn                 int64
dtype: object


MISSING VALUES
CustomerID           0
Age                  0
Gender               0
Tenure               0
Usage Frequency      0
Support Calls        0
Payment Delay        0
Subscription Type    0
Contract Length      0
Total Spend          0
Last Interaction     0
Churn                0
dtype: int64


DUPLICATE ROWS
0


CHURN VALUES
Churn
0    33881
1    30493
Name: count, dtype: int64


UNIQUE VALUES IN CATEGORICAL COLUMNS

Gender:
['Female' 'Male']

Subscription Type:
['Basic' 'Standard' 'Premium']

Contract Length:
['Monthly' 'Annual' 'Quarterly']


In [4]:
# Separate input features (X) and target (y)

X = df.drop(columns=["Churn", "CustomerID"])
y = df["Churn"]

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nFeatures:")
print(X.columns.tolist())

print("\nTarget:")
print(y.name)


X shape: (64374, 10)
y shape: (64374,)

Features:
['Age', 'Gender', 'Tenure', 'Usage Frequency', 'Support Calls', 'Payment Delay', 'Subscription Type', 'Contract Length', 'Total Spend', 'Last Interaction']

Target:
Churn


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)


Training data: (51499, 10)
Testing data: (12875, 10)


In [6]:
# Automatically identify numerical and categorical features

numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)


Numerical features:
['Age', 'Tenure', 'Usage Frequency', 'Support Calls', 'Payment Delay', 'Total Spend', 'Last Interaction']

Categorical features:
['Gender', 'Subscription Type', 'Contract Length']


In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            StandardScaler(),
            numerical_features
        ),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

print("Preprocessing pipeline created successfully.")


Preprocessing pipeline created successfully.


In [8]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    random_state=42,
    eval_metric="logloss"
)

print("XGBoost model created successfully.")


XGBoost model created successfully.


In [9]:
#%pip install xgboost


In [10]:
#Connect preprocessing + XGBoost
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("preprocessor",preprocessor),
    ("model", model)
])

print("Complete ML pipeline created successfully.")


Complete ML pipeline created successfully.


In [11]:
print("Training model...")

pipeline.fit(X_train, y_train)

print("Model training completed successfully.")


Training model...
Model training completed successfully.


In [12]:
#Evaluate the model
from sklearn.metrics import accuracy_score, roc_auc_score

# Make predictions on unseen test data
y_pred = pipeline.predict(X_test)

# Get probability of churn
y_probability = pipeline.predict_proba(X_test)[:, 1]

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_probability)

print("Model Performance")
print("=" * 40)
print(f"Accuracy : {accuracy:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")


Model Performance
Accuracy : 0.9997
ROC-AUC  : 1.0000


In [13]:
# -----------------------------------------
# Model Sanity Check
# -----------------------------------------

from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# 1. Check target distribution
print("Target distribution:")
print(y.value_counts())
print()

# 2. Check whether CustomerID is excluded
print("Features used by the model:")
print(X.columns.tolist())
print()

# 3. Dummy baseline
dummy = DummyClassifier(
    strategy="most_frequent"
)

dummy.fit(X_train, y_train)

dummy_pred = dummy.predict(X_test)

print(
    "Dummy Accuracy:",
    accuracy_score(y_test, dummy_pred)
)

print()

# 4. Logistic Regression baseline
logistic_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

logistic_pipeline.fit(X_train, y_train)

logistic_pred = logistic_pipeline.predict(X_test)
logistic_probability = (
    logistic_pipeline.predict_proba(X_test)[:, 1]
)

print(
    "Logistic Regression Accuracy:",
    accuracy_score(
        y_test,
        logistic_pred
    )
)

print(
    "Logistic Regression ROC-AUC:",
    roc_auc_score(
        y_test,
        logistic_probability
    )
)


Target distribution:
Churn
0    33881
1    30493
Name: count, dtype: int64

Features used by the model:
['Age', 'Gender', 'Tenure', 'Usage Frequency', 'Support Calls', 'Payment Delay', 'Subscription Type', 'Contract Length', 'Total Spend', 'Last Interaction']

Dummy Accuracy: 0.5262912621359224

Logistic Regression Accuracy: 0.8270291262135923
Logistic Regression ROC-AUC: 0.9030001434419446


In [14]:
# Check average feature values for customers who churned vs. did not churn

print(df.groupby("Churn")[[
    "Age",
    "Tenure",
    "Usage Frequency",
    "Support Calls",
    "Payment Delay",
    "Total Spend",
    "Last Interaction"
]].mean().round(2))


         Age  Tenure  Usage Frequency  Support Calls  Payment Delay  \
Churn                                                                 
0      41.13   28.83            16.04            4.5          12.45   
1      42.90   35.52            14.01            6.4          22.33   

       Total Spend  Last Interaction  
Churn                                 
0           560.54             15.52  
1           519.34             15.47  


In [15]:
# Check churn rate for each category

for column in categorical_features:
    print(f"\n{column}")
    print("=" * 40)
    
    result = df.groupby(column)["Churn"].agg(
        ["count", "mean"]
    )
    
    result["churn_rate_%"] = (result["mean"] * 100).round(2)
    
    print(result.drop(columns=["mean"]))



Gender
        count  churn_rate_%
Gender                     
Female  34353         55.05
Male    30021         38.58

Subscription Type
                   count  churn_rate_%
Subscription Type                     
Basic              21451         48.28
Premium            21421         46.50
Standard           21502         47.33

Contract Length
                 count  churn_rate_%
Contract Length                     
Annual           21410         46.22
Monthly          22130         51.61
Quarterly        20834         44.05


In [16]:
# Check the minimum and maximum values of numerical features
# for churned and non-churned customers

for column in numerical_features:
    print(f"\n{column}")
    print("=" * 40)
    
    result = df.groupby("Churn")[column].agg(
        ["min", "max", "mean"]
    ).round(2)
    
    print(result)



Age
       min  max   mean
Churn                 
0       18   65  41.13
1       18   65  42.90

Tenure
       min  max   mean
Churn                 
0        1   60  28.83
1        1   60  35.52

Usage Frequency
       min  max   mean
Churn                 
0        1   30  16.04
1        1   30  14.01

Support Calls
       min  max  mean
Churn                
0        0   10   4.5
1        0   10   6.4

Payment Delay
       min  max   mean
Churn                 
0        0   30  12.45
1        0   30  22.33

Total Spend
       min   max    mean
Churn                   
0      100  1000  560.54
1      100  1000  519.34

Last Interaction
       min  max   mean
Churn                 
0        1   30  15.52
1        1   30  15.47


In [17]:
# Get feature importance from the trained XGBoost model

trained_model = pipeline.named_steps["model"]

importance = trained_model.feature_importances_

print("Number of processed features:", len(importance))
print("Total importance:", importance.sum())


Number of processed features: 15
Total importance: 1.0000001


In [18]:
import joblib

model_path = "../ml/churn_model.pkl"

joblib.dump(pipeline, model_path)

print(f"Model saved successfully to: {model_path}")


Model saved successfully to: ../ml/churn_model.pkl


In [19]:
import joblib

# Load the saved model
loaded_pipeline = joblib.load("../ml/churn_model.pkl")

# Take one customer from the test dataset
sample_customer = X_test.iloc[[0]]

# Make prediction
prediction = loaded_pipeline.predict(sample_customer)[0]
probability = loaded_pipeline.predict_proba(sample_customer)[0][1]

print("Saved model loaded successfully!")
print("Prediction:", prediction)
print(f"Churn probability: {probability:.2%}")


Saved model loaded successfully!
Prediction: 0
Churn probability: 0.02%


In [20]:
# -----------------------------------------
# Train/Test Leakage Investigation
# -----------------------------------------

# 1. Check for duplicate feature rows across train and test
train_features = X_train.copy()
test_features = X_test.copy()

# Convert categorical values to strings so comparison is consistent
train_features = train_features.astype(str)
test_features = test_features.astype(str)

train_rows = set(
    train_features.astype(str).agg("|".join, axis=1)
)

test_rows = set(
    test_features.astype(str).agg("|".join, axis=1)
)

overlap = train_rows.intersection(test_rows)

print("Duplicate feature rows shared by train and test:")
print(len(overlap))

print()

# 2. Check whether identical feature combinations have conflicting labels
feature_columns = X.columns.tolist()

duplicate_features = (
    df.groupby(feature_columns)["Churn"]
    .nunique()
)

conflicting_rows = duplicate_features[
    duplicate_features > 1
]

print("Feature combinations with conflicting Churn labels:")
print(len(conflicting_rows))

print()

# 3. Check exact duplicate complete rows
print("Duplicate complete rows in dataset:")
print(df.duplicated().sum())


Duplicate feature rows shared by train and test:
0

Feature combinations with conflicting Churn labels:
0

Duplicate complete rows in dataset:
0


In [ ]:

# Feature Importance


import pandas as pd

model = pipeline.named_steps["model"]

feature_names = (
    pipeline
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": model.feature_importances_
})

importance = importance.sort_values(
    "Importance",
    ascending=False
)

print(importance.head(15))


                                    Feature  Importance
4                  numerical__Payment Delay    0.383598
7                categorical__Gender_Female    0.133074
3                  numerical__Support Calls    0.121457
13     categorical__Contract Length_Monthly    0.107243
0                            numerical__Age    0.064863
1                         numerical__Tenure    0.061518
2                numerical__Usage Frequency    0.060766
5                    numerical__Total Spend    0.040032
12      categorical__Contract Length_Annual    0.015561
9      categorical__Subscription Type_Basic    0.007516
14   categorical__Contract Length_Quarterly    0.002938
6               numerical__Last Interaction    0.001032
10   categorical__Subscription Type_Premium    0.000403
8                  categorical__Gender_Male    0.000000
11  categorical__Subscription Type_Standard    0.000000
